# SentencePiece: Understanding Subword Tokenization

## What is SentencePiece?

**SentencePiece is a tokenization library/framework**, not a single tokenization algorithm.

SentencePiece can use different tokenization models:

- **Unigram Language Model**
- **BPE (Byte Pair Encoding)**
- Character-level tokenization
- Word-level tokenization

The two important subword algorithms we will study are:

### BPE
BPE starts with small units and repeatedly **merges the most frequent adjacent pieces**.

**Idea:**

small pieces → merge → larger pieces

### Unigram
Unigram starts with a large set of possible pieces and gradually **removes less useful pieces using a probabilistic model**.

**Idea:**

many candidate pieces → remove → final vocabulary

So remember:

> **SentencePiece = Library**  
> **BPE / Unigram = Tokenization algorithms**

In [2]:
# Install the SentencePiece library
!pip install -q sentencepiece

# Import SentencePiece
import sentencepiece as spm

print("SentencePiece installed successfully!")

SentencePiece installed successfully!


## 2. Get a Raw Text Corpus

SentencePiece does not start with a predefined list of tokens.

It **learns its vocabulary from a training corpus**.

A corpus is simply a collection of text used to train the tokenizer.

In this example, we use `botchan.txt`, an English text file from the
SentencePiece repository.

SentencePiece can work directly with **raw, untokenized text**.

For example, the training data can contain:

    I saw a girl with a telescope.
    The weather is beautiful today.
    Tokenization is important in NLP.

We do **not** have to manually split these sentences into words before
giving them to SentencePiece.

In [4]:
import urllib.request

# URL of the training corpus
corpus_url = "https://raw.githubusercontent.com/google/sentencepiece/master/data/botchan.txt"

# Name of the local file
corpus_path = "botchan.txt"

# Download the corpus
urllib.request.urlretrieve(corpus_url, corpus_path)

# Read the corpus
with open(corpus_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Number of lines:", len(lines))

print("\nFirst 5 lines:\n")

for line in lines[:5]:
    print(repr(line))

Number of lines: 4288

First 5 lines:

"\ufeffProject Gutenberg's Botchan (Master Darling), by Kin-nosuke Natsume\n"
'This eBook is for the use of anyone anywhere at no cost and with\n'
'almost no restrictions whatsoever.  You may copy it, give it away or\n'
're-use it under the terms of the Project Gutenberg License included\n'
'with this eBook or online at www.gutenberg.org\n'


## 3. How SentencePiece Handles Spaces

One important feature of SentencePiece is that it treats **whitespace as part
of the text**.

SentencePiece represents a space using the special symbol:

    ▁

This symbol is called the **whitespace marker**.

For example:

    I love NLP

can internally be represented approximately as:

    ▁I▁love▁NLP

Therefore, tokens may look like:

    ["▁I", "▁love", "▁NLP"]

The `▁` tells us that a new word begins after a space.

For example:

    ▁playing

means:

    " playing"

where the `▁` represents the space before `playing`.

This is different from WordPiece, where `##` usually indicates that a token
is a continuation of the previous token.

Example:

    WordPiece:
    ["play", "##ing"]

    SentencePiece:
    ["▁play", "ing"]

SentencePiece therefore keeps whitespace information inside its token
representation, which makes reconstruction of the original text easier.

In [5]:
text = "I love NLP"

print("Original text:")
print(text)

# Demonstration of SentencePiece's whitespace idea
visible_spaces = text.replace(" ", "▁")

print("\nSpaces represented using ▁:")
print("▁" + visible_spaces)

# Convert back
decoded = ("▁" + visible_spaces).replace("▁", " ").strip()

print("\nConverted back to normal text:")
print(decoded)

Original text:
I love NLP

Spaces represented using ▁:
▁I▁love▁NLP

Converted back to normal text:
I love NLP


## 4. Train a SentencePiece Unigram Model

Now we will actually train a tokenizer.

We will use the **Unigram Language Model** inside the SentencePiece library.

Remember:

    SentencePiece = library
    Unigram       = tokenization algorithm

The basic idea of Unigram is:

    Start with many possible subword pieces
                    ↓
    Assign probabilities to the pieces
                    ↓
    Determine which pieces are useful for representing the corpus
                    ↓
    Remove less useful pieces
                    ↓
    Keep the final vocabulary

So, conceptually:

    LARGE candidate vocabulary
              ↓
           pruning
              ↓
    SMALLER final vocabulary

This is different from BPE:

    BPE:
    small pieces → merge → larger pieces

    Unigram:
    many candidate pieces → remove → final vocabulary

In [6]:
spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix="unigram_model",
    vocab_size=5500,
    model_type="unigram",
    character_coverage=1.0,
)

print("Unigram model training completed!")

Unigram model training completed!


## 5. Load the Trained Unigram Model

Training created two important files:

    unigram_model.model
    unigram_model.vocab

The `.model` file contains the trained SentencePiece tokenizer.

The `.vocab` file contains the learned vocabulary pieces and their scores.

Now we load the trained model using:

    SentencePieceProcessor

The processor can then:

- Convert text → subword pieces
- Convert text → token IDs
- Convert token IDs → original text

The complete flow is:

    Original Text
         ↓
    SentencePiece
         ↓
    Subword Pieces
         ↓
    Token IDs

And decoding performs the reverse:

    Token IDs
         ↓
    SentencePiece
         ↓
    Original Text

In [7]:
# Load the trained Unigram SentencePiece model

sp_unigram = spm.SentencePieceProcessor(
    model_file="unigram_model.model"
)

print("Vocabulary size:", sp_unigram.get_piece_size())

print("<unk> ID :", sp_unigram.unk_id())
print("<s> ID   :", sp_unigram.bos_id())
print("</s> ID  :", sp_unigram.eos_id())
print("<pad> ID :", sp_unigram.pad_id())

Vocabulary size: 5500
<unk> ID : 0
<s> ID   : 1
</s> ID  : 2
<pad> ID : -1


## 6. Convert a Sentence into SentencePiece Tokens

Now let's tokenize a sentence using our trained Unigram model.

Example:

    I saw a girl with a telescope.

SentencePiece does two important things:

1. It represents word boundaries using `▁`
2. It divides the text into pieces learned during training

For example, the result may look approximately like:

    ["▁I", "▁saw", "▁a", "▁girl", "▁with", "▁a", "▁telescope", "."]

However, the exact segmentation depends on the vocabulary learned from the
training corpus.

A rare word may therefore be divided into multiple pieces.

For example:

    tokenization

could conceptually become:

    ["▁token", "ization"]

The important point is:

> SentencePiece does not follow a fixed manual rule for splitting every word.
> The pieces come from the vocabulary learned during training.

In [9]:
sample_sentence = "I saw a girl with a telescope."

# Convert text into SentencePiece pieces
pieces = sp_unigram.encode(
    sample_sentence,
    out_type=str
)

# Convert the same text into token IDs
ids = sp_unigram.encode(
    sample_sentence,
    out_type=int
)

print("Original sentence:")
print(sample_sentence)

print("\nSentencePiece pieces:")
print(pieces)

print("\nToken IDs:")
print(ids)

Original sentence:
I saw a girl with a telescope.

SentencePiece pieces:
['▁I', '▁saw', '▁a', '▁girl', '▁with', '▁a', '▁t', 'el', 'es', 'c', 'op', 'e', '.']

Token IDs:
[6, 257, 10, 1104, 21, 10, 536, 946, 409, 957, 1546, 135, 4]


## 7. What Does `▁` Mean in the Output?

Suppose SentencePiece produces:

    ["▁I", "▁saw", "▁a", "▁girl"]

The symbol:

    ▁

represents a whitespace / word boundary.

Therefore:

    ▁I      → " I"
    ▁saw    → " saw"
    ▁a      → " a"
    ▁girl   → " girl"

A piece without `▁` usually continues the current word.

For example:

    ["▁token", "ization"]

means:

    " token" + "ization"

which reconstructs:

    " tokenization"

Therefore:

    ▁token      → start of a new word
    ization     → continuation of that word

This is one of the main ideas behind SentencePiece.

In [10]:
# Decode token IDs back into text

decoded = sp_unigram.decode(ids)

print("Token IDs:")
print(ids)

print("\nDecoded text:")
print(decoded)

print("\nMatches original:")
print(decoded == sample_sentence)

Token IDs:
[6, 257, 10, 1104, 21, 10, 536, 946, 409, 957, 1546, 135, 4]

Decoded text:
I saw a girl with a telescope.

Matches original:
True


## 8. Manual Decoding — Understanding What SentencePiece Does

We can also approximately understand decoding manually.

Suppose our pieces are:

    ["▁I", "▁saw", "▁a", "▁girl"]

### Step 1: Join all pieces

    ▁I▁saw▁a▁girl

### Step 2: Replace `▁` with normal spaces

     I saw a girl

### Step 3: Remove the extra leading space

    I saw a girl

So the basic idea is:

    Pieces
       ↓
    Join pieces
       ↓
    Replace ▁ with space
       ↓
    Original text

SentencePiece's `decode()` function performs the proper decoding automatically.

In [11]:
# Manual demonstration of decoding

manual_decode = "".join(pieces)

print("1. Joined pieces:")
print(manual_decode)

manual_decode = manual_decode.replace("▁", " ")

print("\n2. Replace ▁ with spaces:")
print(repr(manual_decode))

manual_decode = manual_decode.strip()

print("\n3. Remove leading/trailing space:")
print(manual_decode)

1. Joined pieces:
▁I▁saw▁a▁girl▁with▁a▁telescope.

2. Replace ▁ with spaces:
' I saw a girl with a telescope.'

3. Remove leading/trailing space:
I saw a girl with a telescope.
